# KV cache and inference optimization

The KV cache makes generation fast by remembering the keys and values already computed.

During decoding, previously computed keys and values can be reused. That saves repeated compute, but memory grows with layers, heads, context length, and batch size. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(97)


def softmax(logits):
    values = np.asarray(logits, dtype=float)
    shifted = values - values.max(axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)


def kv_ladder():
    return [
        {"name": "D1 one prompt", "length": 4, "new_tokens": 1, "layers": 2, "heads": 2, "dim": 3, "correct_prefix": True},
        {"name": "D2 few-shot prompt batch", "length": 6, "new_tokens": 2, "layers": 2, "heads": 2, "dim": 4, "correct_prefix": True},
        {"name": "D3 distractor prefix reuse", "length": 7, "new_tokens": 2, "layers": 3, "heads": 2, "dim": 4, "correct_prefix": False},
        {"name": "D4 tiny chat snippets", "length": 10, "new_tokens": 3, "layers": 4, "heads": 4, "dim": 6, "correct_prefix": True},
        {"name": "D5 longer context memory stress", "length": 24, "new_tokens": 5, "layers": 6, "heads": 4, "dim": 8, "correct_prefix": False},
    ]


def make_cache(length, layers=2, heads=2, dim=3):
    keys = RNG.normal(size=(layers, heads, length, dim))
    values = RNG.normal(size=(layers, heads, length, dim))
    return keys, values

## The concept, built once: cached attention

At decoding step $t$, cached keys and values let the model compute only the new query:
$$\mathrm{Attn}(q_t,K_{1:t},V_{1:t})=\mathrm{softmax}(q_tK_{1:t}^\top/\sqrt{d_k})V_{1:t}.$$
The cache is exact only for the same prompt prefix and compatible positions.

In [ ]:
def cached_attention(query, keys, values, new_key=None, new_value=None):
    scores = query @ keys.T / math.sqrt(keys.shape[-1])
    weights = softmax(scores)
    output = weights @ values
    if new_key is not None and new_value is not None:
        keys = np.vstack([keys, new_key.reshape(1, -1)])
        values = np.vstack([values, new_value.reshape(1, -1)])
    return {
        "weights": weights,
        "output": output,
        "keys": keys,
        "values": values,
    }


def recompute_states(length):
    return length * (length + 1) // 2


def cached_states(length):
    return length


def kv_memory_bytes(layers, heads, length, dim, bytes_per_value=2):
    tensors = 2
    return layers * heads * tensors * length * dim * bytes_per_value

The lesson exact checks are no-cache states $1+2+3+4+5=15$, cache states 5, fp16 KV memory $2\times2\times2\times4\times3\times2=192$ bytes, scores $[2,1,0]$ softmax to $[0.665,0.245,0.090]$, and appending one token grows length $4\to5$.

In [ ]:
query = np.array([1.0, 0.0, 0.0])
keys = np.eye(3)
values = np.eye(3)
built = cached_attention(query, keys, values, new_key=np.ones(3), new_value=np.ones(3))
score_weights = softmax(np.array([2.0, 1.0, 0.0]))
no_cache = recompute_states(5)
with_cache = cached_states(5)
memory = kv_memory_bytes(2, 2, 4, 3)

assert no_cache == 15
assert with_cache == 5
assert memory == 192
assert np.allclose(np.round(score_weights, 3), np.array([0.665, 0.245, 0.090]))
assert built["keys"].shape[0] == 4

print("no-cache states", no_cache)
print("cached states", with_cache)
print("KV memory bytes", memory)
print("attention weights", np.round(score_weights, 3))
print("cache length after append", built["keys"].shape[0])

## The dataset ladder

D1-D5 is an inline prompt/context ladder: one prompt, few-shot batches, wrong-prefix reuse, real-ish chat snippets, and longer contexts that stress memory.

In [ ]:
ladder = kv_ladder()
for rung in ladder:
    memory = kv_memory_bytes(rung["layers"], rung["heads"], rung["length"], rung["dim"])
    print(rung["name"], "length", rung["length"], "new tokens", rung["new_tokens"], "memory bytes", memory, "same prefix", rung["correct_prefix"])

In [ ]:
rows = []
for rung in ladder:
    final_length = rung["length"] + rung["new_tokens"]
    no_cache_cost = recompute_states(final_length) - recompute_states(rung["length"])
    cache_cost = rung["new_tokens"]
    memory = kv_memory_bytes(rung["layers"], rung["heads"], final_length, rung["dim"])
    correctness = 1.0 if rung["correct_prefix"] else 0.0
    cost_per_token = cache_cost / rung["new_tokens"]
    rows.append({
        "name": rung["name"],
        "context": final_length,
        "no_cache": no_cache_cost,
        "cache": cache_cost,
        "memory": memory,
        "correctness": correctness,
        "cost_per_token": cost_per_token,
    })
for row in rows:
    print(f"{row['name']}: context={row['context']} no_cache={row['no_cache']} cache={row['cache']} memory={row['memory']} correct={row['correctness']:.0f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, row in enumerate(rows):
    axes[0, idx].bar(["cache", "memory/1000"], [row["cache"], row["memory"] / 1000], color=["navy", "gray"])
    axes[0, idx].set_title(f"D{idx + 1} growth")
    axes[1, idx].bar(["correct"], [row["correctness"]], color="navy")
    axes[1, idx].set_ylim(0, 1)
fig.suptitle("KV-cache growth panels and correctness")
plt.figure(figsize=(6, 3))
plt.plot([row["context"] for row in rows], [row["cache"] for row in rows], marker="o", label="cached cost")
plt.plot([row["context"] for row in rows], [row["no_cache"] for row in rows], marker="s", label="no-cache extra cost")
plt.xlabel("context length")
plt.ylabel("computed token states")
plt.legend()
plt.grid(True)

## Pitfall on D5: memory growth and wrong-prefix caches

KV memory grows linearly in context length, and reusing a cache from a different prefix produces plausible numbers for the wrong conversation. The fix keys caches by exact prefix and caps context when memory exceeds budget.

In [ ]:
hard = ladder[-1]
wrong_prefix_attention = softmax(np.array([2.0, 1.0, 0.0]))
wrong_correctness = 0.0
prefix_key_a = tuple(["campaign", "quality", "fell"])
prefix_key_b = tuple(["campaign", "budget", "rose"])
cache_store = {prefix_key_a: "cache_A"}
retrieved = cache_store.get(prefix_key_b)
capped_length = min(hard["length"], 16)
capped_memory = kv_memory_bytes(hard["layers"], hard["heads"], capped_length, hard["dim"])
original_memory = kv_memory_bytes(hard["layers"], hard["heads"], hard["length"], hard["dim"])
print("wrong-prefix weights look plausible", np.round(wrong_prefix_attention, 3), "correctness", wrong_correctness)
print("exact-prefix cache lookup", retrieved)
print("original memory", original_memory)
print("capped memory", capped_memory)

## Evaluate it + practice

            - Metric: cost per generated token and correctness; compare to the no-skill baseline, recompute the full prefix each step.
            - Cheap sanity check: cache length grows by one row per generated token.
            - Ablation: reuse a cache under a different prefix and mark correctness zero.
            - Failure signals: memory exceeds budget or exact-prefix lookup misses.
            - Practice: Compute KV memory for L=8, H=8, T=128, d=64.
- Practice: Plot cached versus uncached state counts for 20 tokens.
- Practice: Design a prefix key that includes position settings.